In [1]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import soundfile as sf

# ==============================================================================
# 1. GPU ENABLED FBANK EXTRACTOR
# ==============================================================================
class FBankExtractor:
    def __init__(self, device, sample_rate=16000, n_mels=80, n_fft=400, hop_length=160):
        self.sample_rate = sample_rate
        self.device = device

        # Khởi tạo transform MelSpectrogram trực tiếp trên GPU
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, 
            n_fft=n_fft, 
            n_mels=n_mels, 
            hop_length=hop_length, 
            center=False
        ).to(self.device)

    def _apply_cmvn(self, feature):
        """Áp dụng chuẩn hóa Mean and Variance Normalization"""
        mean = feature.mean(dim=-1, keepdim=True)
        std = feature.std(dim=-1, keepdim=True)
        return (feature - mean) / (std + 1e-6)

    def extract(self, waveform_cpu):
        # Đưa dữ liệu lên GPU
        waveform_gpu = waveform_cpu.to(self.device)
        if waveform_gpu.dim() == 1: 
            waveform_gpu = waveform_gpu.unsqueeze(0)

        # Tính toán FBank (Log-Mel Spectrogram)
        mel_spec = self.mel_transform(waveform_gpu)
        fbank = torch.log(mel_spec + 1e-6)

        # Chuẩn hóa
        fbank = self._apply_cmvn(fbank)
        
        return fbank.squeeze(0) # Trả về tensor (n_mels, time)

# ==============================================================================
# 2. DATASET (LOGIC RESUME & SCAN)
# ==============================================================================
class FBankDataset(Dataset):
    def __init__(self, root_dir, output_dir, extractor, sample_rate=16000):
        self.root_dir = root_dir
        self.output_dir = output_dir
        self.extractor = extractor
        self.sample_rate = sample_rate
        self.file_list = []
        
        print(f"-> Đang quét thư mục đầu vào: {root_dir}")
        
        # Quét tệp và kiểm tra Resume logic
        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.wav', '.flac', '.mp3')):
                    input_path = os.path.join(root, file)
                    
                    # Tính toán đường dẫn file .pt sẽ lưu
                    rel_path = os.path.relpath(input_path, root_dir)
                    pt_filename = os.path.splitext(rel_path)[0] + ".pt"
                    expected_path = os.path.join(output_dir, pt_filename)

                    if not os.path.exists(expected_path):
                        self.file_list.append(input_path)

        print(f"-> Tổng số tệp mới cần xử lý: {len(self.file_list)}")

    def __len__(self): 
        return len(self.file_list)

    def __getitem__(self, idx):
        path = self.file_list[idx]
        try:
            wav_numpy, sr = sf.read(path)
            waveform = torch.from_numpy(wav_numpy).float()
            
            # Chuẩn hóa channel
            if waveform.dim() == 1: 
                waveform = waveform.unsqueeze(0)
            else: 
                waveform = waveform.t()
            
            if waveform.shape[0] > 1: 
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            # Resample nếu cần
            if sr != self.sample_rate:
                resampler = T.Resample(sr, self.sample_rate)
                waveform = resampler(waveform)

            # Trích xuất FBank (trả về tensor trên GPU hoặc CPU tùy thiết lập)
            fbank = self.extractor.extract(waveform)
            
            return fbank.cpu(), path 
        except Exception as e:
            print(f" Lỗi xử lý {path}: {e}")
            return None, None

# ==============================================================================
# 3. EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # --- CẤU HÌNH ---
    INPUT_PATH = r"D:\my_project\SLP301-data\train\train_vi_5s"
    OUTPUT_PATH = r"D:\my_project\SLP301-data\train\train_vi_5s\FBank"
    BATCH_SIZE = 1 # Tăng giảm tùy vào VRAM của bạn
    
    # --- THIẾT BỊ ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🔥 Đang chạy trên: {device}")

    extractor = FBankExtractor(device=device)
    dataset = FBankDataset(INPUT_PATH, OUTPUT_PATH, extractor)

    if len(dataset) == 0:
        print("\n✅ Tất cả tệp đã được xử lý xong!")
        exit()

    # Dùng num_workers=0 khi trích xuất audio trên Windows để tránh lỗi multiprocessing
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"\nBắt đầu trích xuất...")
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    for batch in tqdm(loader):
        fbanks, paths = batch
        if fbanks is None: continue

        for i in range(len(paths)):
            # Tạo đường dẫn lưu trữ giữ nguyên cấu trúc thư mục gốc
            rel_path = os.path.relpath(paths[i], INPUT_PATH)
            save_path = os.path.join(OUTPUT_PATH, os.path.splitext(rel_path)[0] + ".pt")
            
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            
            # Lưu tensor FBank (n_mels, time)
            torch.save(fbanks[i], save_path)

    print(f"\n Hoàn thành! Dữ liệu lưu tại: {OUTPUT_PATH}")

🔥 Đang chạy trên: cuda
-> Đang quét thư mục đầu vào: D:\my_project\SLP301-data\train\train_vi_5s
-> Tổng số tệp mới cần xử lý: 1

Bắt đầu trích xuất...


100%|██████████| 1/1 [00:00<00:00,  1.68it/s]


 Hoàn thành! Dữ liệu lưu tại: D:\my_project\SLP301-data\train\train_vi_5s\FBank


# Packing into big files

In [2]:
import os
import torch
import glob
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_shard_fbank(ptm_path, fbank_map, save_dir):
    """Hàm xử lý độc lập cho 1 file Shard FBank"""
    
    # Tạo tên file đầu ra (tự thay full_hubert và hubert thành fbank)
    shard_name = os.path.basename(ptm_path)
    fbank_shard_name = shard_name.replace("full_hubert", "fbank").replace("hubert", "fbank").replace("wavlm", "fbank").replace("wav2vec2", "fbank")
    out_path = os.path.join(save_dir, fbank_shard_name)
    
    # Double check (phòng hờ)
    if os.path.exists(out_path):
        return fbank_shard_name

    # Mở file PTM để lấy index
    ptm_data = torch.load(ptm_path, map_location='cpu')
    
    features = []
    speaker_ids = []
    filenames = []
    
    for fname, spk_id in zip(ptm_data['filenames'], ptm_data['speaker_ids']):
        pt_name = os.path.splitext(fname)[0] + ".pt"
        
        if pt_name in fbank_map:
            try:
                # Thử load file
                feat = torch.load(fbank_map[pt_name], map_location='cpu')
                features.append(feat.half())  # Ép kiểu float16 cho nhẹ RAM
                speaker_ids.append(spk_id)
                filenames.append(fname)
            except RuntimeError as e:
                # Bắt lỗi file hỏng và in ra chính xác đường dẫn
                print(f"\n❌ [CẢNH BÁO] File bị hỏng (corrupted): {fbank_map[pt_name]}")
                # Tạm thời bỏ qua file lỗi này để tiến trình không bị dừng
                continue 
        else:
            print(f"⚠ Thiếu file FBank: {pt_name}")
            
    torch.save({
        'features': features,
        'speaker_ids': speaker_ids,
        'filenames': filenames
    }, out_path)
    
    return fbank_shard_name

def pack_fbank_features_fast(ptm_shard_dir, fbank_dir, save_dir, num_threads=8):
    os.makedirs(save_dir, exist_ok=True)
    ptm_shards = sorted(glob.glob(os.path.join(ptm_shard_dir, "*.pt")))
    
    # ==========================================================
    # LỌC CÁC SHARD CHƯA ĐƯỢC ĐÓNG GÓI (SKIP NHỮNG CÁI ĐÃ XONG)
    # ==========================================================
    pending_shards = []
    for p in ptm_shards:
        shard_name = os.path.basename(p)
        fbank_shard_name = shard_name.replace("full_hubert", "fbank").replace("hubert", "fbank").replace("wavlm", "fbank").replace("wav2vec2", "fbank")
        out_path = os.path.join(save_dir, fbank_shard_name)
        
        if not os.path.exists(out_path):
            pending_shards.append(p)
            
    if not pending_shards:
        print(f"⏭️ BỎ QUA: Toàn bộ {len(ptm_shards)} Shards FBank đã được đóng gói từ trước!")
        return
        
    print(f"🔍 Cần đóng gói {len(pending_shards)}/{len(ptm_shards)} Shards FBank còn thiếu.")
    # ==========================================================
    
    print(f"🔍 Đang quét các file FBank lẻ tại:\n   {fbank_dir}")
    fbank_files = glob.glob(os.path.join(fbank_dir, "**", "*.pt"), recursive=True)
    fbank_map = {os.path.basename(f): f for f in fbank_files}
    print(f"✅ Đã tìm thấy {len(fbank_files)} file FBank.")
    
    print(f"🚀 Bắt đầu đóng gói với {num_threads} luồng ổ cứng song song...")
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(process_single_shard_fbank, p, fbank_map, save_dir) for p in pending_shards]
        for future in tqdm(as_completed(futures), total=len(pending_shards), desc="Đóng gói FBank Shards"):
            future.result() 
            
    print(f"🎉 Đã lưu shard FBank thành công vào:\n   {save_dir}") 


# =====================================================================
# CẤU HÌNH ĐƯỜNG DẪN CHO FBANK
# =====================================================================
BASE_PTM = r"D:\my_project\SLP301-data\embeddings"
BASE_HC = r"D:\my_project\SLP301-data\train"
BASE_OUT = r"D:\my_project\SLP301-data\train"

FOLDER_MAPPING = {
    "FBank": "fbank_shards",
    "Only MFBE": "mfbe_shards",
    "Only Pitch": "pitch_shards",
    "MFBE + Pitch": "mfbe_pitch_shards",
    "Only MFCC": "mfcc_shards",
    "MFCC + Pitch": "mfcc_pitch_shards"
}

TASKS = {
    # "train_raw": ["FBank"],
    # "train_vi_full": ["FBank"],
    # "train_vi_7s": ["Only MFCC", "MFCC + Pitch"],
    # "train_vi_3s": ["Only MFCC", "MFCC + Pitch"],
    "train_vi_5s": ["FBank"]
}

# =====================================================================
# BẮT ĐẦU CHUỖI TIẾN TRÌNH FBANK
# =====================================================================
total_tasks = sum(len(subfolders) for subfolders in TASKS.values())
current_task = 1

for dataset, subfolders in TASKS.items():
    # Xử lý tên PTM folder cho khớp
    ptm_folder_name = "full_hubert_shards" if dataset == "train_vi_full" else "hubert_shards"
    ptm_dir = os.path.join(BASE_PTM, dataset, ptm_folder_name)
    
    for sub_f in subfolders:
        fbank_dir = os.path.join(BASE_HC, dataset, sub_f)
        out_dir = os.path.join(BASE_OUT, dataset, FOLDER_MAPPING[sub_f])
        
        print("\n" + "="*80)
        print(f"⏳ ĐANG XỬ LÝ [{current_task}/{total_tasks}]: Data [{dataset}] -> Đặc trưng [{sub_f}]")
        print(f"   ➤ Lấy PTM từ: {ptm_dir}")
        print(f"   ➤ Lấy FBank từ: {fbank_dir}")
        print(f"   ➤ Lưu Shard tới: {out_dir}")
        print("="*80)
        
        pack_fbank_features_fast(ptm_dir, fbank_dir, out_dir, num_threads=8)
        current_task += 1
        
print("\n" + "🏆"*20)
print("XUẤT SẮC! ĐÃ HOÀN THÀNH ĐÓNG GÓI TOÀN BỘ ĐẶC TRƯNG FBANK!")
print("🏆"*20)


⏳ ĐANG XỬ LÝ [1/1]: Data [train_vi_5s] -> Đặc trưng [FBank]
   ➤ Lấy PTM từ: D:\my_project\SLP301-data\embeddings\train_vi_5s\hubert_shards
   ➤ Lấy FBank từ: D:\my_project\SLP301-data\train\train_vi_5s\FBank
   ➤ Lưu Shard tới: D:\my_project\SLP301-data\train\train_vi_5s\fbank_shards
🔍 Cần đóng gói 1/40 Shards FBank còn thiếu.
🔍 Đang quét các file FBank lẻ tại:
   D:\my_project\SLP301-data\train\train_vi_5s\FBank
✅ Đã tìm thấy 394182 file FBank.
🚀 Bắt đầu đóng gói với 8 luồng ổ cứng song song...


Đóng gói FBank Shards: 100%|██████████| 1/1 [00:12<00:00, 12.85s/it]

🎉 Đã lưu shard FBank thành công vào:
   D:\my_project\SLP301-data\train\train_vi_5s\fbank_shards

🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
XUẤT SẮC! ĐÃ HOÀN THÀNH ĐÓNG GÓI TOÀN BỘ ĐẶC TRƯNG FBANK!
🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆


In [3]:
import os
import torch
import glob
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_shard_fbank(ptm_path, fbank_map, save_dir):
    """Hàm xử lý độc lập cho 1 file Shard FBank"""
    
    # Tạo tên file đầu ra (tự thay full_hubert và hubert thành fbank)
    shard_name = os.path.basename(ptm_path)
    fbank_shard_name = shard_name.replace("full_hubert", "fbank").replace("hubert", "fbank").replace("wavlm", "fbank").replace("wav2vec2", "fbank")
    out_path = os.path.join(save_dir, fbank_shard_name)
    
    # Double check (phòng hờ)
    if os.path.exists(out_path):
        return fbank_shard_name

    # Mở file PTM để lấy index
    ptm_data = torch.load(ptm_path, map_location='cpu')
    
    features = []
    speaker_ids = []
    filenames = []
    
    for fname, spk_id in zip(ptm_data['filenames'], ptm_data['speaker_ids']):
        # Tránh lỗi tuple bằng
        pt_name = os.path.splitext(fname)[0] + ".pt"
        
        if pt_name in fbank_map:
            feat = torch.load(fbank_map[pt_name], map_location='cpu')
            features.append(feat.half())  # Ép kiểu float16 cho nhẹ RAM
            speaker_ids.append(spk_id)
            filenames.append(fname)
        else:
            print(f"⚠ Thiếu file FBank: {pt_name}")
            
    torch.save({
        'features': features,
        'speaker_ids': speaker_ids,
        'filenames': filenames
    }, out_path)
    
    return fbank_shard_name

def pack_fbank_features_fast(ptm_shard_dir, fbank_dir, save_dir, num_threads=8):
    os.makedirs(save_dir, exist_ok=True)
    ptm_shards = sorted(glob.glob(os.path.join(ptm_shard_dir, "*.pt")))
    
    # ==========================================================
    # LỌC CÁC SHARD CHƯA ĐƯỢC ĐÓNG GÓI (SKIP NHỮNG CÁI ĐÃ XONG)
    # ==========================================================
    pending_shards = []
    for p in ptm_shards:
        shard_name = os.path.basename(p)
        fbank_shard_name = shard_name.replace("full_hubert", "fbank").replace("hubert", "fbank").replace("wavlm", "fbank").replace("wav2vec2", "fbank")
        out_path = os.path.join(save_dir, fbank_shard_name)
        
        if not os.path.exists(out_path):
            pending_shards.append(p)
            
    if not pending_shards:
        print(f"⏭️ BỎ QUA: Toàn bộ {len(ptm_shards)} Shards FBank đã được đóng gói từ trước!")
        return
        
    print(f"🔍 Cần đóng gói {len(pending_shards)}/{len(ptm_shards)} Shards FBank còn thiếu.")
    # ==========================================================
    
    print(f"🔍 Đang quét các file FBank lẻ tại:\n   {fbank_dir}")
    fbank_files = glob.glob(os.path.join(fbank_dir, "**", "*.pt"), recursive=True)
    fbank_map = {os.path.basename(f): f for f in fbank_files}
    print(f"✅ Đã tìm thấy {len(fbank_files)} file FBank.")
    
    print(f"🚀 Bắt đầu đóng gói với {num_threads} luồng ổ cứng song song...")
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(process_single_shard_fbank, p, fbank_map, save_dir) for p in pending_shards]
        for future in tqdm(as_completed(futures), total=len(pending_shards), desc="Đóng gói FBank Shards"):
            future.result() 
            
    print(f"🎉 Đã lưu shard FBank thành công vào:\n   {save_dir}") 


# =====================================================================
# CẤU HÌNH ĐƯỜNG DẪN CHO FBANK
# =====================================================================
BASE_PTM = r"D:\my_project\SLP301-data\embeddings"
BASE_HC = r"D:\my_project\SLP301-data\train"
BASE_OUT = r"D:\my_project\SLP301-data\train"

FOLDER_MAPPING = {
    "FBank": "fbank_shards",
    "Only MFBE": "mfbe_shards",
    "Only Pitch": "pitch_shards",
    "MFBE + Pitch": "mfbe_pitch_shards",
    "Only MFCC": "mfcc_shards",
    "MFCC + Pitch": "mfcc_pitch_shards"
}

TASKS = {
    # "train_raw": ["FBank"],
    # "train_vi_full": ["FBank"],
    # "train_vi_7s": ["Only MFCC", "MFCC + Pitch"],
    "train_vi_3s": ["Only MFCC", "MFCC + Pitch"],
    # "train_vi_5s": ["FBank"]
}

# =====================================================================
# BẮT ĐẦU CHUỖI TIẾN TRÌNH FBANK
# =====================================================================
total_tasks = sum(len(subfolders) for subfolders in TASKS.values())
current_task = 1

for dataset, subfolders in TASKS.items():
    # Xử lý tên PTM folder cho khớp
    ptm_folder_name = "full_hubert_shards" if dataset == "train_vi_full" else "hubert_shards"
    ptm_dir = os.path.join(BASE_PTM, dataset, ptm_folder_name)
    
    for sub_f in subfolders:
        fbank_dir = os.path.join(BASE_HC, dataset, sub_f)
        out_dir = os.path.join(BASE_OUT, dataset, FOLDER_MAPPING[sub_f])
        
        print("\n" + "="*80)
        print(f"⏳ ĐANG XỬ LÝ [{current_task}/{total_tasks}]: Data [{dataset}] -> Đặc trưng [{sub_f}]")
        print(f"   ➤ Lấy PTM từ: {ptm_dir}")
        print(f"   ➤ Lấy FBank từ: {fbank_dir}")
        print(f"   ➤ Lưu Shard tới: {out_dir}")
        print("="*80)
        
        pack_fbank_features_fast(ptm_dir, fbank_dir, out_dir, num_threads=8)
        current_task += 1
        
print("\n" + "🏆"*20)
print("XUẤT SẮC! ĐÃ HOÀN THÀNH ĐÓNG GÓI TOÀN BỘ ĐẶC TRƯNG FBANK!")
print("🏆"*20)


⏳ ĐANG XỬ LÝ [1/2]: Data [train_vi_3s] -> Đặc trưng [Only MFCC]
   ➤ Lấy PTM từ: D:\my_project\SLP301-data\embeddings\train_vi_3s\hubert_shards
   ➤ Lấy FBank từ: D:\my_project\SLP301-data\train\train_vi_3s\Only MFCC
   ➤ Lưu Shard tới: D:\my_project\SLP301-data\train\train_vi_3s\mfcc_shards
🔍 Cần đóng gói 56/56 Shards FBank còn thiếu.
🔍 Đang quét các file FBank lẻ tại:
   D:\my_project\SLP301-data\train\train_vi_3s\Only MFCC
✅ Đã tìm thấy 559876 file FBank.
🚀 Bắt đầu đóng gói với 8 luồng ổ cứng song song...


Đóng gói FBank Shards: 100%|██████████| 56/56 [10:01<00:00, 10.75s/it] 


🎉 Đã lưu shard FBank thành công vào:
   D:\my_project\SLP301-data\train\train_vi_3s\mfcc_shards

⏳ ĐANG XỬ LÝ [2/2]: Data [train_vi_3s] -> Đặc trưng [MFCC + Pitch]
   ➤ Lấy PTM từ: D:\my_project\SLP301-data\embeddings\train_vi_3s\hubert_shards
   ➤ Lấy FBank từ: D:\my_project\SLP301-data\train\train_vi_3s\MFCC + Pitch
   ➤ Lưu Shard tới: D:\my_project\SLP301-data\train\train_vi_3s\mfcc_pitch_shards
🔍 Cần đóng gói 56/56 Shards FBank còn thiếu.
🔍 Đang quét các file FBank lẻ tại:
   D:\my_project\SLP301-data\train\train_vi_3s\MFCC + Pitch
✅ Đã tìm thấy 559876 file FBank.
🚀 Bắt đầu đóng gói với 8 luồng ổ cứng song song...


Đóng gói FBank Shards: 100%|██████████| 56/56 [12:00<00:00, 12.87s/it] 

🎉 Đã lưu shard FBank thành công vào:
   D:\my_project\SLP301-data\train\train_vi_3s\mfcc_pitch_shards

🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
XUẤT SẮC! ĐÃ HOÀN THÀNH ĐÓNG GÓI TOÀN BỘ ĐẶC TRƯNG FBANK!
🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
